In [ ]:
import constants as c
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.integrate import solve_ivp

vk = c.vk
vrf = c.vrf
z_offset = c.z0
xy1k = c.xy1k
xy2k = c.xy2k
pi = math.pi
um = c.um
cm = c.cm
mm = c.mm
K = c.K
e = c.e

eps = 1
alpha = 1/137.06
d_chain = 3.7066438742e-06  # 3.7 µm separation between ions
delta_ion_2 = 0#-4.9903032463e-14 # small displacement of ion 2
m_dm = 1e-27
m_ion = c.m 
Z_charge = c.Z
freq_x = 719430.7131391969
freq_y = 3031200.0099101723
freq_z = 3002153.5607483205
omega = 2*pi*np.array([freq_x,freq_y,freq_z])
hbar = 1.0545718e-34 
mode_y = 2*pi*np.array([3031200.01,2944587.06,2818861.50])
mode_z = 2*pi*np.array([3002153.56,2914677.59,2787603.39])

In [ ]:
# RF pseudo-force
def FRF(xyz,m,Z,q=c.e,omrf=c.omega,VRF=c.vrf,ymin=c.y11,yedge1=c.y21,yedge2=c.y12,ymax=c.y22):
    xyz = np.asarray(xyz, dtype=float)
    x = xyz[:, 0]
    y = xyz[:, 1]
    z = xyz[:, 2]
    m = float(np.asarray(m).ravel()[0])
    Z = float(np.asarray(Z).ravel()[0])
    q = float(np.asarray(q).ravel()[0])
    omrf = float(np.asarray(omrf).ravel()[0])
    VRF = float(np.asarray(VRF).ravel()[0])
    #this is the pseudo potential, which is Z^2*(Div[PhiRF]/cos(om*t))^2/(4m*omega^2)
    divypart=divatan(z,yedge2-y)-divatan(z,yedge1-y)+divatan(z,ymin-y)-divatan(z,ymax-y)
    divzpart=divatan(yedge2-y,z)-divatan(yedge1-y,z)+divatan(ymin-y,z)-divatan(ymax-y,z) 
    divyparty=-2*divypart*(divatandown(z,yedge2-y)-divatandown(z,yedge1-y)+divatandown(z,ymin-y)-divatandown(z,ymax-y)) 
    divypartz=2*divypart*(divatanup(z,yedge2-y)-divatanup(z,yedge1-y)+divatanup(z,ymin-y)-divatanup(z,ymax-y)) 
    divzparty=-2*divzpart*(divatanup(yedge2-y,z)-divatanup(yedge1-y,z)+divatanup(ymin-y,z)-divatanup(ymax-y,z)) 
    divzpartz=2*divzpart*(divatandown(yedge2-y,z)-divatandown(yedge1-y,z)+divatandown(ymin-y,z)-divatandown(ymax-y,z)) 
    return -1*((VRF*Z*q)/(2*pi*np.sqrt(m)*omrf))**2*np.column_stack([divypartz*0, divzparty+divyparty, divzpartz+divypartz])

# DC force
def divatan(up,down):
    #This is d(arctan2(up,down))/ddown up to a minus sign. It's useful for the pseudo-potential
    return up/(up**2+down**2)
def divatanup(up,down):
    #This is d(divatan(up,down))/dup
    return (down**2-up**2)/(up**2+down**2)**2
def divatandown(up,down):
    #This is d(divatan(up,down))/ddown
    return -2*up*down/(up**2+down**2)**2

def anatangrad(xi,yi,xyz,v): # gradient term of DC potential
    xyz = np.asarray(xyz, dtype=float)
    x = xyz[:, 0]
    y = xyz[:, 1]
    z = xyz[:, 2]
    dy=y-yi
    dx=x-xi
    r = np.sqrt(dx**2+dy**2+z**2); # added distance
    dry2=z**2+dy**2
    drx2=z**2+dx**2
    divy=z*dx/(r*dry2); # divide by factor r
    divz=-dy*dx*(1/dry2+1/drx2)/r; # divide by factor r
    divx=z*dy/(r*drx2); # divide by factor r
    return (v/(2*np.pi))*np.column_stack([divx, divy, divz])

def FDC_single(x1,y1,x2,y2,xyz,v,Z):
    return -Z*e * (anatangrad(x2,y2,xyz,v)-anatangrad(x2,y1,xyz,v)-anatangrad(x1,y2,xyz,v)+anatangrad(x1,y1,xyz,v))

def FDC(xyz, m, Z, q=c.e, omrf=c.omega,
        ymin=c.y11, yedge1=c.y21, yedge2=c.y12, ymax=c.y22):
    xyz = np.asarray(xyz, dtype=float)
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("xyz must have shape (n, 3)")
    z_original = xyz[:, 2].copy()
    force = np.zeros_like(xyz)
    for k in range(0, 40):
        xyz_k = xyz.copy()
        if (k != 19 and k != 39):
            xyz_k[:, 2] = z_original - z_offset  # keep scalar math
        else:
            xyz_k[:, 2] = z_original
        (x1k, y1k) = xy1k[k]
        (x2k, y2k) = xy2k[k]
        # enforce scalar on vk[k]
        v_k = float(np.asarray(vk[k]).ravel()[0])
        force_single = FDC_single(x1k, y1k, x2k, y2k, xyz_k, v_k, Z)
        # ensure force_single is shape (3,)
        force_single = np.asarray(force_single, dtype=float)
    if force_single.shape != xyz.shape:
        raise ValueError(
            f"FDC_single returned unexpected shape {force_single.shape}; "
            f"expected {xyz.shape} at k={k}"
        )
    force += force_single
    return force

# Full Simulation Code

In [6]:
def run_simulation_many_dm_single_ion(
    x0_dms, v0_dms, x0_ion, v0_ion,
    t_span, dt, run_rutherford,
    rtol=1e-13, atol=1e-16
):
    """
    Run a single-ion / many-DM trajectory simulation.

    This version includes:
        - one trapped ion near the origin
        - many DM particles
        - ion-DM Coulomb forces
        - no DM-DM Coulomb forces

    Parameters
    ----------
    x0_dms : array-like, shape (n_dm, 3)
        Initial positions of the DM particles.

    v0_dms : array-like, shape (n_dm, 3)
        Initial velocities of the DM particles.

    x0_ion : array-like, shape (3,)
        Initial position of the single ion.

    v0_ion : array-like, shape (3,)
        Initial velocity of the single ion.

    t_span : tuple (t_start, t_end)
        Start and end time of the integration.

    dt : float
        Time increment used to build the evaluation grid.

    run_rutherford : bool
        If True, set trap fields to zero for Rutherford scattering comparison.

    Returns
    -------
    t_eval : ndarray, shape (N,)
        Times at which the solution was evaluated.

    x_dms_sol : ndarray, shape (n_dm, 3, N)
        DM position trajectories.

    v_dms_sol : ndarray, shape (n_dm, 3, N)
        DM velocity trajectories.

    x_ion_sol : ndarray, shape (3, N)
        Ion position trajectory.

    v_ion_sol : ndarray, shape (3, N)
        Ion velocity trajectory.
    """

    x0_dms = np.asarray(x0_dms, dtype=float)
    v0_dms = np.asarray(v0_dms, dtype=float)
    x0_ion = np.asarray(x0_ion, dtype=float)
    v0_ion = np.asarray(v0_ion, dtype=float)

    if x0_dms.ndim != 2 or x0_dms.shape[1] != 3:
        raise ValueError("x0_dms must have shape (n_dm, 3)")

    if v0_dms.shape != x0_dms.shape:
        raise ValueError("v0_dms must have the same shape as x0_dms")

    if x0_ion.shape != (3,):
        raise ValueError("x0_ion must have shape (3,)")

    if v0_ion.shape != (3,):
        raise ValueError("v0_ion must have shape (3,)")

    n_dm = x0_dms.shape[0]

    # The single ion is trapped about the origin.
    r_ion_eq = np.zeros(3, dtype=float)

    def rhs_unified(t, U, r_ion_eq, use_harmonic_ion=True):
        """
        State vector layout:

        U = [
            x_ion(3),
            x_dms(3*n_dm),
            v_ion(3),
            v_dms(3*n_dm)
        ]
        """

        idx0 = 0
        idx1 = idx0 + 3
        idx2 = idx1 + 3 * n_dm
        idx3 = idx2 + 3
        idx4 = idx3 + 3 * n_dm

        x_ion = U[idx0:idx1]
        x_dms = U[idx1:idx2].reshape((n_dm, 3))

        v_ion = U[idx2:idx3]
        v_dms = U[idx3:idx4].reshape((n_dm, 3))

        dx_iondt = v_ion
        dx_dmsdt = v_dms

        dv_iondt = np.zeros(3)
        dv_dmsdt = np.zeros((n_dm, 3))

        # --------------------------------------------------
        # Trap field acting on ion
        # --------------------------------------------------
        if use_harmonic_ion:
            if run_rutherford:
                dv_iondt += np.zeros(3)
            else:
                dv_iondt += -omega**2 * (x_ion - r_ion_eq)
        else:
            if run_rutherford:
                E_i = np.zeros(3)
            else:
                dc_i = np.asarray(FDC(*x_ion))
                rf_i = np.asarray(FRF(*x_ion))
                E_i = (dc_i + rf_i) / e

            dv_iondt += (Z * e / m_ion) * E_i

        # --------------------------------------------------
        # Trap field acting on each DM particle
        # --------------------------------------------------
        for k in range(n_dm):
            x_dm = x_dms[k]

            if run_rutherford:
                E_dm = np.zeros(3)
            else:
                dc_dm = np.asarray(FDC(*x_dm))
                rf_dm = np.asarray(FRF(*x_dm))
                E_dm = (dc_dm + rf_dm) / e

            dv_dmsdt[k] += (eps * e / m_dm) * E_dm

        # --------------------------------------------------
        # Coulomb interactions: ion <-> each DM particle
        #
        # No DM-DM Coulomb forces are included.
        # --------------------------------------------------
        for k in range(n_dm):
            x_dm = x_dms[k]

            delta = x_ion - x_dm
            r = np.linalg.norm(delta)

            force = K * eps * e**2 * delta / r**3

            # Force on ion due to DM k
            dv_iondt += force / m_ion

            # Equal and opposite force on DM k due to ion
            dv_dmsdt[k] -= force / m_dm

        return np.concatenate([
            dx_iondt,
            dx_dmsdt.flatten(),
            dv_iondt,
            dv_dmsdt.flatten()
        ])

    print("Number of DM particles:", n_dm)
    print("Initial ion position:", x0_ion)
    print("Initial ion velocity:", v0_ion)

    # Initial state
    U0 = np.concatenate([
        x0_ion,
        x0_dms.flatten(),
        v0_ion,
        v0_dms.flatten()
    ]).astype(float)

    # Time grid
    t_eval = np.arange(t_span[0], t_span[1], dt)

    # Solve
    sol = solve_ivp(
        rhs_unified,
        t_span,
        U0,
        method="DOP853",
        t_eval=t_eval,
        dense_output=True,
        rtol=rtol,
        atol=atol,
        args=(r_ion_eq,)
    )

    print("Success?", sol.success)
    print("Message:", sol.message)

    if sol.t.size > 0:
        print("Final time:", sol.t[-1])

    # Extract solution
    idx0 = 0
    idx1 = idx0 + 3
    idx2 = idx1 + 3 * n_dm
    idx3 = idx2 + 3
    idx4 = idx3 + 3 * n_dm

    x_ion_sol = sol.y[idx0:idx1]
    x_dms_flat_sol = sol.y[idx1:idx2]

    v_ion_sol = sol.y[idx2:idx3]
    v_dms_flat_sol = sol.y[idx3:idx4]

    # Convert flattened DM solution into shape (n_dm, 3, N)
    n_t = sol.y.shape[1]

    x_dms_sol = x_dms_flat_sol.reshape((n_dm, 3, n_t))
    v_dms_sol = v_dms_flat_sol.reshape((n_dm, 3, n_t))

    return t_eval, x_ion_sol, v_ion_sol

In [5]:
def analyze_simulation_results(x0_dms, v0_dms, x0_ion, v0_ion, t_span, dt, t_min, fit_curve, rutherford, show_plots,
                                rtol=1e-13, atol=1e-16):
    if(rutherford):
        t00, xion00, vion00 = run_simulation_many_dm_single_ion(x0_dms, v0_dms, x0_ion, v0_ion, t_span, dt, run_rutherford=True, rtol=rtol, atol=atol)
    else:
        t00, xion00, vion00 = run_simulation_many_dm_single_ion(x0_dms, v0_dms, x0_ion, v0_ion, t_span, dt, run_rutherford=False, rtol=rtol, atol=atol)
    t0 = t00/um
    xion0 = xion00/um
    vion0 = vion00
    x0_ion1, y0_ion1, z0_ion1 = xion0[0], xion0[1], xion0[2] # motion of center ion

    ## Plots 
    if(show_plots):

        # -------------------------
        # Ion coordinates vs time
        # -------------------------
        plt.figure(figsize=(18, 5))

        plt.subplot(1, 3, 1)
        plt.plot(t0, x0_ion1, label="x ion")
        plt.xlabel("t (us)")
        plt.ylabel("x (um)")
        plt.title("Ion x-position vs time")
        plt.legend()

        plt.subplot(1, 3, 2)
        plt.plot(t0, y0_ion1, label="y ion")
        plt.xlabel("t (us)")
        plt.ylabel("y (um)")
        plt.title("Ion y-position vs time")
        plt.legend()

        plt.subplot(1, 3, 3)
        plt.plot(t0, z0_ion1, label="z ion")
        plt.xlabel("t (us)")
        plt.ylabel("z (um)")
        plt.title("Ion z-position vs time")
        plt.legend()

        plt.tight_layout()
        plt.show()

    return t00, xion00, vion00